## PCA + Metadata

In [5]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

from scipy.stats import ttest_ind

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"


### Carregando o dataset

In [6]:
DATASET_PATH = "../../data/processed/dataset_features.parquet"

df_dataset = pd.read_parquet(DATASET_PATH)
df_dataset.shape

(108, 409)

In [7]:
#  Age, Gender and condition
# Create a dictionary where keys are the metric names (e.g., 'degree')
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "local_efficiency_",
    "hub_frequency_",
    "weighted_degree_"
]

NODE_FEATURES = [
    c for c in df_dataset.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]
df_dataset = df_dataset.rename(columns={
    "Gender_ 1=female_2=male": "gender",
    "Age": "age",
    "Handedness": "handedness",
    "Education": "education",
    "DRUG_0=negative_1=Positive": "drug",
    "Smoking_num_(Non-smoker=1, Occasional Smoker=2, Smoker=3)": "smoking",
    "SKID_Diagnoses": "skid_diagnoses",
    "SKID_Diagnoses 1": "skid_diagnoses_1",
    "SKID_Diagnoses 2": "skid_diagnoses_2",
    "Comments_SKID_assessment": "comments_skid_assessment",
    "Hamilton_Scale": "hamilton_scale",
    "BSL23_sumscore": "bsl23_sumscore",
    "BSL23_behavior": "bsl23_behavior",
    "AUDIT": "audit",
    "Standard_Alcoholunits_Last_28days": "standard_alcoholunits_last_28days",
    "Alcohol_Dependence_In_1st-3rd_Degree_relative": "alcohol_dependence_in_1st_3rd_degree_relative",
    "Relationship_Status": "relationship_status"
})

In [8]:

X = df_dataset[NODE_FEATURES]
meta = df_dataset[["condition", "age", "gender"]]


imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
pca = PCA(n_components=2)

X_imputed = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_imputed)
X_pca = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
df_pca = pd.concat([df_pca, meta.reset_index(drop=True)], axis=1)
df_pca

,PC1,PC2,condition,age,gender
0,-13.881348,-1.488678,EC,20-25,2
1,-6.802801,3.357945,EO,20-25,2
2,-3.856589,6.184554,EC,20-25,2
3,-14.926402,0.039579,EO,20-25,2
4,6.154506,7.903362,EC,25-30,1
...,...,...,...,...,...
103,-12.564063,1.732716,EO,20-25,2
104,0.100491,4.633810,EC,30-35,2
105,1.166146,-0.852626,EO,30-35,2
106,-5.902983,5.551366,EC,20-25,2


In [9]:

color_vars = ["condition", "age", "gender"]

for var in color_vars:
    fig = px.scatter(
        df_pca,
        x="PC1",
        y="PC2",
        color=var,
        title=f"PCA colored by {var}",
        opacity=0.8,
        template="plotly_dark"
    )
    fig.show()


A projeção por PCA revelou ampla sobreposição entre os grupos EC e EO nos dois primeiros componentes principais, não indicando separação visual evidente entre as condições com base nas métricas de rede selecionadas. Da mesma forma, não foi observado um padrão claro de agrupamento associado à faixa etária ou ao gênero. Foram identificados alguns participantes com escores extremos, sugerindo perfis de conectividade distintos que devem ser investigados individualmente.

In [ ]:
pca.explained_variance_ratio_, pca.explained_variance_ratio_.sum()

(array([0.46294088, 0.05664618]), np.float64(0.5195870593355701))

: 